# Haan warm-start (self-contained)

이 노트북 **하나만** 업로드하면 실행됩니다. 별도 파일이 필요 없습니다.

무엇을 하는가 — **백본만 Qwen3로 교체하고, 오디오/Depth는 Moshi 가중치를 그대로** 얹어 Haan 모델을 조립한다 (ARCHITECTURE 1 / 5.4.1 / 5.4.2):

```
model.* / lm_head / self_attn.{q,k}_norm   <- Qwen3      (백본 교체)
embed_tokens.{0..K-1}                       <- Moshi user 측  (5.4.2)
depth_decoder.*                             <- Moshi 그대로   (5.4.1)
```

- Moshi 는 `latentforge/transformers-moshi` fork 에서 설치.
- Haan 모델 코드(configuration/modeling/generation/processing + warm-start)는 아래 셀에 그대로 담겨 있어, 실행 시 로컬 패키지로 기록된다.
- 실제 체크포인트(`Qwen/Qwen3-8B`, `kmhf/hf-moshiko`)를 읽는다. 첫 실행 시 Qwen3(~16GB)를 받으며, 조립된 9B 모델은 bf16 로 ~18GB.

**위에서 아래로 순서대로 실행하세요.**

## 1. Moshi fork 설치

stock transformers 에도 `models/moshi/` 는 있지만 `generation_moshi.py` 는 fork 에만 있으므로, 그 파일로 판별한다.

In [ ]:
import importlib.util, importlib, pathlib, subprocess, sys

FORK   = "git+https://github.com/latentforge/transformers-moshi.git"
MARKER = "models/moshi/generation_moshi.py"      # fork 에만 존재

def fork_marker():
    try:
        spec = importlib.util.find_spec("transformers")
    except Exception:
        return None
    if spec is None or not spec.origin:
        return None
    return pathlib.Path(spec.origin).parent / MARKER

def fork_installed():
    p = fork_marker()
    return p is not None and p.exists()

already_imported = "transformers" in sys.modules
if fork_installed():
    print("fork 확인:", fork_marker())
else:
    print("stock transformers 감지 (또는 미설치) -> fork 설치 중... 수 분 걸립니다.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", FORK])
    importlib.invalidate_caches()
    if not fork_installed():
        raise RuntimeError(f"설치 후에도 fork 를 찾지 못했습니다. 기대 경로: {fork_marker()}")
    print("설치 완료:", fork_marker())
    if already_imported:
        print("\n*** 이미 예전 transformers 를 import 했습니다. 런타임 재시작 후 이 셀부터 다시 실행하세요. ***")
        raise SystemExit

import transformers, torch
from transformers.models.moshi.generation_moshi import MoshiGenerationMixin   # fork 확정
print("transformers", transformers.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 2. Haan 패키지 기록

소스가 노트북 안에 있어, 아래 셀들이 로컬 디렉토리에 쓴다.

In [ ]:
import pathlib, sys
root = pathlib.Path("project_amnesty")
(root / "models" / "haan").mkdir(parents=True, exist_ok=True)
(root / "utils").mkdir(parents=True, exist_ok=True)
(root / "__init__.py").write_text('"""Haan (notebook copy)."""\n')
(root / "utils" / "__init__.py").write_text('"""utils (notebook copy)."""\n')
(root / "models" / "__init__.py").write_text("from project_amnesty.models.haan import *  # noqa: F401,F403\n")
if "" not in sys.path:
    sys.path.insert(0, "")
print("패키지 트리 생성 완료")

In [ ]:
%%writefile project_amnesty/models/haan/configuration_haan.py
"""Haan model configuration -- subclasses the Moshi config from `transformers`.

Haan IS a Moshi variant, so both configs inherit Moshi's and add only the fields the
Haan deltas need. Everything else (audio_vocab_size, the Mimi sub-config, the depth
sub-config wiring, backbone dims) is Moshi's.

Grounded in docs/contexts/ARCHITECTURE.md:
  - 3.3   Role Token: role is a signal separate from the audio embedding table
  - 5.4   shared Depth: per-codebook params, shared across roles, role added on top
  - 5.4.1 warm-start: audio cardinality 2048 and depth dim 1024 are reused as-is
"""

from huggingface_hub.dataclasses import strict
from transformers.models.moshi.configuration_moshi import MoshiConfig, MoshiDepthConfig

__all__ = ["HaanConfig", "HaanDepthConfig"]


# ARCHITECTURE 3.2 shows why a role signal is required at all: self/user share one summed
# input vector, so without it code 42 from either stream reaches the transformer as the
# same vector. `role_mode` selects how that signal is applied.
#
#   "scale"    -- per-role elementwise gain on each audio embedding (default)
#   "additive" -- ARCHITECTURE 3.3 as literally written: one learned vector per role,
#                 added to the embedding sum
#
# "additive" is degenerate **on the Temporal side only**, and is kept there so the ablation in
# ARCHITECTURE 3.6 / 6.2 can run it. See `RoleEmbedding` in modeling_haan.py for the derivation;
# the short version is that adding a constant to a symmetric sum leaves the sum invariant under
# swapping the two streams.
#
# The Depth Transformer is a different situation and takes a different default. There the role is
# applied to `Proj_k(z_s)` of a row that carries exactly ONE stream (ARCHITECTURE 5.4's batch-2
# layout), not to a sum over both -- so distinct role vectors give distinct rows and nothing
# collapses. ARCHITECTURE 5.4.1 writes that step additively:
#
#     z_depth[k, role] = depformer_in_k(z_s) + RoleEmb[role]
#
# and 5.4 gives four reasons for preferring it there. So `role_mode` (Temporal) defaults to
# "scale" and `depth_role_mode` defaults to "additive": the degeneracy repair is applied where the
# degeneracy is, and the spec is followed where it is sound.
ROLE_MODES = ("scale", "additive")


@strict
class HaanDepthConfig(MoshiDepthConfig):
    """Depth Transformer config (ARCHITECTURE 5.0 eq.2, 5.4).

    `num_codebooks` keeps its Moshi meaning -- how many codebooks the depth decoder
    *predicts*, both streams included (`2 * K`). What Haan changes is that the per-index
    parameters behind those positions are only `K` wide and shared across the two roles,
    so `num_codebooks` must divide evenly by `num_roles`.
    """

    model_type = "haan_depth"

    num_roles: int = 2
    # ARCHITECTURE 5.4.1's formula, `depformer_in_k(z_s) + RoleEmb[role]`. Additive is well-posed
    # here -- each row carries one stream, so the Temporal-side degeneracy does not apply. Set from
    # `HaanConfig.depth_role_mode`, which is a separate knob from the Temporal `role_mode`.
    role_mode: str = "additive"

    @property
    def codebooks_per_role(self) -> int:
        """Width of one audio stream -- the size of the shared per-index parameter axis."""
        return self.num_codebooks // self.num_roles

    def validate_architecture(self):
        """Part of `@strict`-powered validation. Validates the architecture of the config."""
        super().validate_architecture()
        if self.role_mode not in ROLE_MODES:
            raise ValueError(f"`role_mode={self.role_mode!r}` must be one of {ROLE_MODES}.")
        if self.num_roles < 1:
            raise ValueError(f"`num_roles={self.num_roles}` must be at least 1.")
        if self.num_codebooks % self.num_roles:
            raise ValueError(
                f"`num_codebooks={self.num_codebooks}` must be divisible by `num_roles={self.num_roles}`: the "
                "depth decoder's per-index parameters are shared across roles, so every role must cover the same "
                "number of codebooks."
            )


@strict
class HaanConfig(MoshiConfig):
    """Top-level Haan config (ARCHITECTURE 1 / 3 / 4 / 5).

    `num_codebooks` is Moshi's: the width of a *single* audio stream (`K`, 8 for Mimi).
    Haan holds `K` shared audio embedding tables rather than Moshi's `2 * K` separate
    ones, and tells the two streams apart with the role signal instead
    (ARCHITECTURE 3.3).
    """

    model_type = "haan"
    # Resolved through `sub_configs` so the depth config is built as a `HaanDepthConfig`
    # rather than silently falling back to Moshi's.
    sub_configs = {**MoshiConfig.sub_configs, "depth_decoder_config": HaanDepthConfig}

    num_roles: int = 2
    # Temporal-side role signal (ARCHITECTURE 3.3). "scale" because the Temporal input sums both
    # streams and an additive tag cancels out of that sum -- see `ROLE_MODES` above.
    role_mode: str = "scale"
    # Depth-side role signal (ARCHITECTURE 5.4.1). Deliberately a separate knob, and deliberately a
    # different default: the depth rows carry one stream each, so nothing collapses and the spec's
    # additive formula holds. Mirrored onto `depth_decoder_config['role_mode']`.
    depth_role_mode: str = "additive"

    # Full causal attention, where Moshi defaults to a 3000-frame sliding window.
    #
    # This is an override of `MoshiConfig.sliding_window = 3000`, and it is a deliberate
    # architectural choice rather than a tidy-up:
    #
    #   - RISKS 7.7 lists Moshi's "~5 minute context limit" among the teacher weaknesses Haan must
    #     not inherit, and asks that the student be checked for it after KD. Keeping the window
    #     would transplant that limit *structurally* -- not through KD, but by construction, so no
    #     amount of checking afterwards could find it absent.
    #   - RISKS 7.8 warns that a mismatch between the Qwen3 backbone's sequence handling and
    #     "Moshi's frame-rate / window assumptions" is a bug that happens silently at code level.
    #     This is exactly that mismatch: Qwen3 is full-attention over 40960 positions.
    #
    # It is not inert, and it is not loud either. `MoshiModel.forward` selects
    # `create_sliding_window_causal_mask` whenever this is non-None, and `DynamicCache` builds
    # window layers that physically evict KV past the window. But a sliding window of W is
    # bit-identical to full causal for any sequence of length <= W, and training runs at 750 frames
    # (60 s, MOSHI_KD_DATASET_PLAN 166), so every test passes and the divergence appears only in
    # conversations past 240 s.
    #
    # Also note this must be correct BEFORE the first cache is built -- mutating the config later
    # leaves already-constructed cache layers sliding.
    sliding_window: int | None = None

    # QK-Norm (ARCHITECTURE 1: the Helium -> Qwen3 backbone substitution).
    #
    # Qwen3 RMS-normalizes the query and key over the head dimension, per head, immediately before
    # RoPE, and ships a `q_norm`/`k_norm` weight per layer. Moshi (Helium) does not, and
    # `MoshiAttention` has nowhere to put those tensors -- so a Qwen3 backbone loaded into a stock
    # Moshi attention would silently drop them and run at a q/k scale the weights were never
    # trained under. `HaanAttention` adds them; this flag says whether they exist.
    #
    # Defaults to True: ARCHITECTURE 1 makes Qwen3 the backbone, so QK-Norm is Haan's normal shape
    # and the default should describe Haan rather than its Moshi parent.
    #
    # Set it False ONLY for a Moshi-backbone build (the 3.6 baseline ablation, or a Moshi-only
    # warm-start). This is not cosmetic: an RMSNorm whose weight is all ones still rescales its
    # input, so leaving it on under Moshi backbone weights corrupts an otherwise exact transfer,
    # and turning it off under Qwen3 weights discards two tensors per layer.
    use_qk_norm: bool = True

    def __post_init__(self, **kwargs):
        # `depth_role_mode` is mirrored onto the depth sub-config's `role_mode` -- NOT the Temporal `role_mode`,
        # which describes a different mechanism in a different place (see `ROLE_MODES`). `num_roles` is *derived*
        # rather than mirrored, because the word means two different things on the two configs:
        #
        #   here          how many streams share the audio tables and the Depth parameters (ARCH 3.3/5.4) -- 2
        #   depth config  how many streams the depth decoder actually *predicts* (ARCH 5.0.3) -- 1 or 2
        #
        # Those coincide when the depth decoder predicts both streams, and they do not when it predicts only
        # its own (Moshi's released layout, `predict_user_stream=False` in the warm-start). Mirroring `2` onto a
        # self-only depth decoder made `codebooks_per_role` come out as K/2, which silently split one stream in
        # half and labelled its upper codebooks role=1 -- no error, just self cb4..7 treated as the user.
        # Deriving it from the codebook counts makes that unrepresentable.
        #
        # `None` is normalised to `{}` first, exactly as MoshiConfig does for the same reason: left as `None` the
        # depth decoder silently keeps `HaanDepthConfig`'s own defaults instead of what was asked for. An already-built sub-config is normalised back to a dict so
        # it takes the same path -- otherwise it skips this block entirely and `save_pretrained` /
        # `from_pretrained` (which always goes through the dict path) would silently *change* the architecture.
        if self.depth_decoder_config is None:
            self.depth_decoder_config = {}
        elif not isinstance(self.depth_decoder_config, dict):
            self.depth_decoder_config = self.depth_decoder_config.to_dict()
            self.depth_decoder_config.pop("model_type", None)

        # Moshi defaults this to `2 * num_codebooks`; the number of streams is Haan's to decide.
        self.depth_decoder_config.setdefault("num_codebooks", self.num_roles * self.num_codebooks)
        predicted = self.depth_decoder_config["num_codebooks"]
        self.depth_decoder_config.update(
            {
                "role_mode": self.depth_role_mode,
                "num_roles": max(predicted // self.num_codebooks, 1) if self.num_codebooks else self.num_roles,
            }
        )
        super().__post_init__(**kwargs)

    def validate_architecture(self):
        """Part of `@strict`-powered validation. Validates the architecture of the config."""
        super().validate_architecture()
        for field in ("role_mode", "depth_role_mode"):
            value = getattr(self, field)
            if value not in ROLE_MODES:
                raise ValueError(f"`{field}={value!r}` must be one of {ROLE_MODES}.")

        # `num_roles` is not a free knob on the Temporal side: `MoshiForConditionalGeneration.forward`
        # unconditionally builds the audio input as `cat([assistant_audio_codes, user_audio_codes])`, and
        # `generation_haan._embed_audio_codes` reads the role back as `codebook // num_codebooks`. Both are
        # hard-wired to exactly two streams. `num_roles=1` used to validate, and then died on the first
        # forward with `IndexError: index 1 is out of bounds` from `RoleEmbedding` -- a config-shaped bug
        # surfacing as a runtime crash. Rejected here instead.
        if self.num_roles != 2:
            raise ValueError(
                f"`num_roles={self.num_roles}` must be 2: the Temporal input is built by concatenating the "
                "assistant and user streams and the role is recovered from the codebook index, so the "
                "self/user pair is structural rather than configurable (ARCHITECTURE 3.3)."
            )

        # `num_codebooks` is a stream width, so it has to be positive. Without this, `num_codebooks=0`
        # validated cleanly (the axis check below compares 0 != 0 and passes) and `-4` produced
        # `codebooks_per_role=-4` and `max_position_embeddings=-7`, both of which survived a
        # save_pretrained/from_pretrained round trip.
        if self.num_codebooks < 1:
            raise ValueError(f"`num_codebooks={self.num_codebooks}` must be at least 1 (it is a stream width).")

        # The real invariant behind the depth decoder's shared parameter axis: one role covers exactly one
        # stream. `HaanDepthConfig` can only check that its own `num_codebooks` divides by its own
        # `num_roles`, which is a weaker proxy. In practice `__post_init__` derives the depth's `num_roles`
        # from the codebook counts, so the only configuration that actually trips this is a depth
        # `num_codebooks` that is not a positive multiple of the parent's.
        depth = self.depth_decoder_config
        if depth is not None and not isinstance(depth, dict) and depth.codebooks_per_role != self.num_codebooks:
            raise ValueError(
                f"the depth decoder's shared parameter axis is {depth.codebooks_per_role} wide but one audio "
                f"stream is {self.num_codebooks} codebooks (`num_codebooks`). Those must match: the axis is "
                "shared across roles, so each role has to cover exactly one stream. Set "
                f"`depth_decoder_config['num_codebooks']` to a multiple of {self.num_codebooks} "
                f"(got {depth.num_codebooks})."
            )

In [ ]:
%%writefile project_amnesty/models/haan/modeling_haan.py
"""Haan model -- subclasses the Moshi model from `transformers`.

Haan reuses Moshi's whole stack (Temporal + Depth Transformer, generation loop, delay
pattern) and overrides the three places where the architecture actually differs:

  - ARCHITECTURE 3.3  the Temporal input side. Moshi holds `2 * K` separate audio
    embedding tables, one set per stream; Haan holds `K` shared tables and marks the
    stream with a role signal instead. `_embed_audio_codes` is the single hook -- both
    `forward` and the generation loop route audio embedding through it.
  - ARCHITECTURE 5.4  the Depth Transformer. Moshi's per-index parameters run `2 * K`
    wide (one index per predicted codebook, both streams laid end to end); Haan's run
    `K` wide and are shared across roles, with the role added onto the per-index
    projection of `z_s`.
  - ARCHITECTURE 1   attention. Qwen3 replaces Helium as the backbone and adds QK-Norm,
    so `HaanAttention` / `HaanDecoderLayer` / `HaanModel` override Moshi's -- but on the
    *backbone* only. The Depth Transformer keeps Moshi's plain attention.

Everything else -- the delay pattern, `generate` -- is inherited untouched.
"""

import torch
import torch.nn as nn
from transformers.cache_utils import Cache, DynamicCache
from transformers.masking_utils import create_causal_mask
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast
from transformers.integrations.hub_kernels import use_kernelized_func
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.moshi.modeling_moshi import (
    MoshiAttention,
    MoshiDecoderLayer,
    MoshiDepthDecoderForCausalLM,
    MoshiDepthDecoderModel,
    MoshiFlexibleLinear,
    MoshiForConditionalGeneration,
    MoshiModel,
    MoshiRMSNorm,
    apply_rotary_pos_emb,
    eager_attention_forward,
    get_codebook_idx,
)
from transformers.processing_utils import Unpack
from transformers.utils import TransformersKwargs, can_return_tuple
from transformers.utils.generic import merge_with_config_defaults
from transformers.utils.output_capturing import capture_outputs

from .configuration_haan import HaanConfig, HaanDepthConfig
from .generation_haan import HaanGenerationMixin

__all__ = [
    "RoleEmbedding",
    "HaanAttention",
    "HaanDecoderLayer",
    "HaanModel",
    "HaanDepthDecoderModel",
    "HaanDepthDecoderForCausalLM",
    "HaanForConditionalGeneration",
]


def _sequence_width(input_ids: torch.Tensor | None, inputs_embeds: torch.Tensor | None) -> int:
    """Sequence length of whichever input was actually given, or 0 if neither was.

    Both are checked because the depth decoder's layout guard has to hold for an `inputs_embeds`
    caller too: gating on `input_ids` alone let a `2 * K`-wide embedding sequence walk the banned
    sequential path with no error, leaking the assistant's frame-internal choice into the user
    logits (measured: a perturbation that must move them by 0 moved them by 3353).
    """
    if input_ids is not None:
        return input_ids.shape[1]
    if inputs_embeds is not None:
        return inputs_embeds.shape[1]
    return 0


class RoleEmbedding(nn.Module):
    """The role signal (ARCHITECTURE 3.3): one learned parameter per role.

    New in Haan -- Moshi has no role signal, because it keeps a separate embedding table
    per stream and so never needs one.

    **On `role_mode="additive"`, which is ARCHITECTURE 3.3 as literally written.** The
    Temporal input at frame `t` is a *sum* over every codebook of both streams
    (ARCHITECTURE 2.1: elementwise sum, not concat). With shared tables that is

        h_t = TextEmb(w_t) + sum_k E_k(A_self[t,k]) + sum_k E_k(A_user[t,k]) + r_self + r_user

    and swapping the two streams leaves `h_t` bit-identical: the code sums commute and
    `r_self + r_user` is a constant that does not depend on which stream is which. So an
    additive role token cannot distinguish "I said X, you said Y" from "I said Y, you
    said X" -- it is not merely a weaker mechanism than Moshi's separate tables
    (ARCHITECTURE 6.2's framing), it carries exactly zero role information at the
    Temporal input. This is the same collapse ARCHITECTURE 3.2 warns about, and adding a
    constant does not avert it.

    `role_mode="scale"` is the minimal repair that keeps everything else about the design
    -- shared tables, two learned vectors, no change to the backbone's RoPE. A per-role
    elementwise gain makes the per-stream contributions `s_self * sum_k E_k(A_self)` and
    `s_user * sum_k E_k(A_user)`, which differ under a swap whenever `s_self != s_user`.
    Both modes are kept so the ablation in ARCHITECTURE 3.6 / 6.2 can measure the gap.

    Both modes initialise to the identity, so a Moshi warm-start (ARCHITECTURE 5.4.1)
    reproduces the teacher's embeddings exactly at step 0. The two roles therefore start
    identical; symmetry breaks on the first step because each role's parameter multiplies
    a different stream's codes and so receives a different gradient.
    """

    def __init__(self, hidden_size: int, num_roles: int = 2, role_mode: str = "scale") -> None:
        super().__init__()
        self.role_mode = role_mode
        self.num_roles = num_roles
        if role_mode == "scale":
            self.role_scale = nn.Parameter(torch.ones(num_roles, hidden_size))
        elif role_mode == "additive":
            self.role_emb = nn.Parameter(torch.zeros(num_roles, hidden_size))
        else:
            raise ValueError(f"`role_mode={role_mode!r}` must be one of ('scale', 'additive').")

    def forward(self, hidden_states: torch.Tensor, role_ids: torch.Tensor | int) -> torch.Tensor:
        """Apply the role signal.

        Args:
            hidden_states (`torch.Tensor` of shape `(..., hidden_size)`):
                Audio embeddings for one role, or a sequence whose positions each carry a role.
            role_ids (`torch.Tensor` or `int`):
                A single role for the whole tensor, or one role per position. A per-position
                tensor of shape `(sequence_length,)` indexes to `(sequence_length, hidden_size)`,
                which broadcasts against a `(batch, sequence_length, hidden_size)` input.
        """
        parameter = self.role_scale if self.role_mode == "scale" else self.role_emb
        selected = parameter[role_ids]
        return hidden_states * selected if self.role_mode == "scale" else hidden_states + selected


@use_kernelized_func(apply_rotary_pos_emb)
class HaanAttention(MoshiAttention):
    """[`MoshiAttention`] plus Qwen3's QK-Norm (ARCHITECTURE 1).

    ARCHITECTURE 1 replaces Helium with Qwen3 as the Temporal backbone, and Qwen3 differs from
    Helium inside attention in exactly one way that carries weights: it RMS-normalizes the query
    and the key over the head dimension, per head, immediately before RoPE.

        query = q_norm(q_proj(x).view(..., heads, head_dim))
        key   = k_norm(k_proj(x).view(..., heads, head_dim))

    `MoshiAttention` has no such module, so loading Qwen3 into it would drop `q_norm.weight` and
    `k_norm.weight` for every layer (72 tensors for Qwen3-8B) and run the remaining weights at a
    q/k scale they were never trained under. That is not a small numerical difference: QK-Norm
    controls the magnitude entering the softmax.

    Gated on `config.use_qk_norm` because the same class also serves the Moshi warm-start, where
    the norms must NOT be present -- an RMSNorm whose weight is all ones still rescales its input,
    so switching it on under Moshi weights would corrupt an otherwise exact transfer. The
    parameters are named `q_norm` / `k_norm` to match Qwen3's checkpoints exactly.
    """

    def __init__(self, config, layer_idx: int | None = None, use_flexible_linear: bool = False):
        super().__init__(config, layer_idx=layer_idx, use_flexible_linear=use_flexible_linear)

        self.use_qk_norm = getattr(config, "use_qk_norm", False)
        if self.use_qk_norm:
            # Over the HEAD dimension only -- not hidden_size. Qwen3 uses its own RMSNorm here;
            # MoshiRMSNorm is the same function (both are T5-style) and keeps this file on one
            # norm implementation.
            self.q_norm = MoshiRMSNorm(self.head_dim, eps=config.rms_norm_eps)
            self.k_norm = MoshiRMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        past_key_values: Cache | None = None,
        codebook_idx: torch.Tensor | None = None,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # NOTE: this is `MoshiAttention.forward` verbatim except for the two `q_norm`/`k_norm`
        # lines marked below. There is no hook between the projections and RoPE, and the norm has
        # to sit exactly there, so the method is copied rather than wrapped -- keep it in sync
        # when the upstream Moshi attention changes.
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states, codebook_idx).view(hidden_shape)
        key_states = self.k_proj(hidden_states, codebook_idx).view(hidden_shape)
        if self.use_qk_norm:
            # <-- the ARCHITECTURE 1 delta. Applied on the (..., heads, head_dim) view, before the
            #     transpose and before RoPE, exactly where Qwen3 applies it. RMSNorm reduces over
            #     the last axis, so norm-then-transpose and transpose-then-norm are identical.
            query_states = self.q_norm(query_states)
            key_states = self.k_norm(key_states)
        query_states = query_states.transpose(1, 2)
        key_states = key_states.transpose(1, 2)
        value_states = self.v_proj(hidden_states, codebook_idx).view(hidden_shape).transpose(1, 2)

        # rotary embeddings are not used in the depth decoder, where `position_embeddings` is None
        if position_embeddings is not None:
            cos, sin = position_embeddings
            query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
            **kwargs,
        )

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output, codebook_idx)
        return attn_output, attn_weights


class HaanDecoderLayer(MoshiDecoderLayer):
    """[`MoshiDecoderLayer`] with [`HaanAttention`] in place of [`MoshiAttention`].

    Everything else -- the MLP, both RMSNorms, the residual structure -- is Moshi's. The attention
    is rebuilt on top of the parent's construction rather than in place of it, so the rest of the
    parent `__init__` stays the single source of truth (the same pattern
    [`HaanForConditionalGeneration`] uses for its embeddings).
    """

    def __init__(self, config, layer_idx: int, use_flexible_linear: bool):
        super().__init__(config, layer_idx, use_flexible_linear)
        self.self_attn = HaanAttention(config, layer_idx=layer_idx, use_flexible_linear=use_flexible_linear)


class HaanModel(MoshiModel):
    """The Haan Temporal Transformer -- [`MoshiModel`] with QK-Norm-capable attention.

    The only structural difference from Moshi is [`HaanDecoderLayer`], and that only matters when
    `config.use_qk_norm` is on (ARCHITECTURE 1's Qwen3 backbone). With it off, this is `MoshiModel`
    parameter for parameter, which is what keeps the Moshi warm-start exact.
    """

    config: HaanConfig

    def __init__(self, config: HaanConfig):
        super().__init__(config)
        self.layers = nn.ModuleList(
            [HaanDecoderLayer(config, layer_idx, use_flexible_linear=False) for layer_idx in range(config.num_hidden_layers)]
        )
        self.post_init()


class HaanDepthDecoderModel(MoshiDepthDecoderModel):
    """The Haan depth decoder (ARCHITECTURE 5.4).

    Same sequence layout as [`MoshiDepthDecoderModel`] -- position `i` is codebook `i` of one
    main-decoder timestep, both streams laid end to end -- but the per-index parameters behind
    those positions are only `codebooks_per_role` wide and shared across roles. A position's
    role is applied to the per-index projection of the main decoder's hidden state:

        z_depth[k, role] = RoleEmb[role](depformer_in[k](z_s))
    """

    config: HaanDepthConfig

    def __init__(self, config: HaanDepthConfig):
        codebooks_per_role = config.codebooks_per_role

        # ARCHITECTURE 5.4.1: "the projections are 8 per k, each shared between the two roles". Every
        # per-index parameter here -- the input tables, `input_projections`, and each decoder layer's
        # flexible linears -- is sized by `config.num_codebooks`, which counts BOTH streams. Haan wants
        # one stream's worth, shared across roles and indexed by `codebook_idx % codebooks_per_role`.
        #
        # So `num_codebooks` is narrowed across the whole of `super().__init__` and restored after,
        # rather than letting the parent build full-width modules here that are then thrown away.
        #
        # This trims the innermost of three build-and-discard levels, not all of them.
        # `MoshiForConditionalGeneration.__init__` and `MoshiDepthDecoderForCausalLM.__init__` still
        # each build a full-width module and drop it, because both hardcode the class they construct
        # and offer no hook; removing those would mean bypassing the parent `__init__` and restating
        # its body, which drifts on every upstream change. The discarded modules are collected --
        # this is transient peak, not a leak.
        #
        # The saving is in peak RSS, and it is worth roughly a third of it at the real depth size
        # (hidden 1024, input 4096, 6 layers, 16 codebooks): ~13.7GB before, ~9GB after. Wall-clock
        # numbers are deliberately not quoted -- they were measured on a loaded machine and did not
        # reproduce; construction time here is dominated by weight init, not by the rebuild.
        #
        # Narrowed in place, not on a `deepcopy`: a copy would leave every layer holding a *different*
        # config object than the one `forward` passes to `create_causal_mask`, and
        # `MoshiAttention.forward` reads `self.config._attn_implementation` at run time -- so the mask
        # and the attention kernel would drift apart. `set_attn_implementation` after construction
        # would then be silently ignored by the layers, `output_attentions=True` would record nothing,
        # and a flex-attention mask would reach an SDPA kernel.
        #
        # Safe because `num_codebooks` is only read while modules are being sized, never in `forward`.
        # `max_position_embeddings` is deliberately not touched: the depth *sequence* still spans both
        # streams, it is only the *parameter* axis that shrinks.
        full_num_codebooks = config.num_codebooks
        config.num_codebooks = codebooks_per_role
        try:
            super().__init__(config)
            self.role_embedding = RoleEmbedding(config.hidden_size, config.num_roles, config.role_mode)
        finally:
            config.num_codebooks = full_num_codebooks

        self.codebooks_per_role = codebooks_per_role

        # The parent's `num_codebooks - 1` input tables (here `K - 1`) are exactly right, and the
        # count is not an oversight to correct. Every row is `[text, cb_0 .. cb_{K-2}]`: the last
        # codebook of a stream is only ever a target, never fed back in. A `K`-th table was briefly
        # added on the reasoning that "under role sharing every slot is an input for some role" --
        # true of the sequential layout, false once `_split_roles` made every row one stream wide.
        # It reached no forward pass, received no gradient, had no warm-start source, and sat at
        # random init.

        self.post_init()

    def _slot_and_role(self, codebook_idx: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Split a position index into (which shared per-index parameter, which role)."""
        return codebook_idx % self.codebooks_per_role, codebook_idx // self.codebooks_per_role

    @merge_with_config_defaults
    @capture_outputs
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        last_hidden_state: torch.FloatTensor | None = None,
        attention_mask: torch.BoolTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        use_cache: bool | None = None,
        position_ids: torch.LongTensor | None = None,
        role_ids: torch.LongTensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPast:
        r"""
        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens. The first element of the sequence must be the text token associated to
            the audio codebooks. The rest of the elements must be flattened audio codebooks.
        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            Sequence of hidden-states at the output of the last layer of the main decoder. Used to contextualize
            `input_ids`.
        role_ids (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Which role each *row* carries, for the batch-parallel rollout of ARCHITECTURE 5.4. When given, the
            sequence covers one stream (`codebooks_per_role` positions) and the role comes from the batch axis.
            When omitted, both streams are laid end to end along the sequence and the role is read off the
            position instead -- the teacher-forced training layout that [`MoshiDepthDecoderModel`] uses.
        """
        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        past_seen_tokens = 0 if past_key_values is None else past_key_values.get_seq_length()
        codebook_idx = get_codebook_idx(input_ids, inputs_embeds, past_seen_tokens)
        slot_idx, role_idx = self._slot_and_role(codebook_idx)
        if role_ids is not None:
            # Validated rather than trusted, because the failure is silent in one direction: indexing
            # a parameter with `-1` is legal Python and aliases to the LAST role, so an upstream
            # convention where -1 means "unknown/padding" would train and generate as role 1 with no
            # error anywhere. A row count that disagrees with the batch broadcasts just as quietly.
            batch = inputs_embeds.shape[0] if input_ids is None else input_ids.shape[0]
            if role_ids.dim() != 1 or role_ids.shape[0] != batch:
                raise ValueError(
                    f"`role_ids` must be one role per row -- shape ({batch},), got {tuple(role_ids.shape)}."
                )
            if not (0 <= role_ids.min() and role_ids.max() < self.config.num_roles):
                raise ValueError(
                    f"`role_ids` must lie in [0, {self.config.num_roles}), got "
                    f"[{int(role_ids.min())}, {int(role_ids.max())}]. Negative ids silently alias to the "
                    "last role rather than raising."
                )
            # Role per row, not per position: index to `(batch, 1, hidden)` so it broadcasts over the sequence.
            role_idx = role_ids[:, None]

        if position_ids is None:
            position_ids = codebook_idx.unsqueeze(0)

        # If inputs_embeds is provided, it has the priority over input_ids, which won't be used
        if inputs_embeds is None:
            # `codebook_idx = arange(seq_len) + past_seen_tokens`, so position `local_idx` in this call
            # is absolute position `past_seen_tokens + local_idx`. Both are already Python ints, so the
            # loop control touches no tensor -- where the previous `for position_idx in codebook_idx:
            # position_idx.item()` forced a GPU->CPU sync every step (8 per training forward, one per
            # generated codebook) and a torch.compile graph break each time. Iterating `range` instead
            # is bit-identical (verified max|Δ|=0 across prefill and every incremental step) and drops
            # those syncs to zero. `codebook_idx` stays a tensor for `_slot_and_role` / `position_ids`,
            # which index it whole and never sync per element.
            inputs_embeds = []
            for local_idx in range(input_ids.shape[1]):
                position_idx = past_seen_tokens + local_idx
                if position_idx == 0:
                    inputs_embeds.append(self.text_embed_tokens(input_ids[:, [local_idx]]))
                else:
                    # Position `p > 0` takes codebook `p - 1` as input, so the shared table is picked by the
                    # *input* codebook's slot, one behind the position's own.
                    slot = (position_idx - 1) % self.codebooks_per_role
                    inputs_embeds.append(self.embed_tokens[slot](input_ids[:, [local_idx]]))

            inputs_embeds = torch.cat(inputs_embeds, dim=1)

        # ARCHITECTURE 5.4: the role rides on the projection of z_s, not on the codebook embedding.
        # Out-of-place (Moshi accumulates in place) so a caller's `inputs_embeds` is never mutated.
        projected = self.role_embedding(self.input_projections(last_hidden_state, slot_idx), role_idx)
        inputs_embeds = inputs_embeds + projected

        causal_mask = create_causal_mask(
            config=self.config,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            position_ids=position_ids,
        )

        hidden_states = inputs_embeds
        for decoder_layer in self.layers:
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=causal_mask,
                past_key_values=past_key_values,
                use_cache=use_cache,
                codebook_idx=slot_idx,
            )

        return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)


class HaanDepthDecoderForCausalLM(MoshiDepthDecoderForCausalLM):
    """The Haan depth decoder with a per-codebook audio language modelling head on top.

    The heads are shared across roles the same way the body is (ARCHITECTURE 5.4).
    """

    config: HaanDepthConfig

    def __init__(self, config: HaanDepthConfig):
        super().__init__(config)
        self.model = HaanDepthDecoderModel(config)
        # One head per codebook of a single stream, shared across roles -- indexed with the same
        # `codebook_idx % codebooks_per_role` the body uses.
        self.codebooks_per_role = config.codebooks_per_role
        self.lm_heads = MoshiFlexibleLinear(config.hidden_size, config.audio_vocab_size, self.codebooks_per_role)
        # How many roles `generate` rolls out. Flipped by `HaanGenerationMixin.set_depth_mode`
        # (ARCHITECTURE 5.4: q16 for simulation, q8 for live conversation).
        self.num_generation_roles = config.num_roles
        self.post_init()

    @torch.no_grad()
    def generate(self, last_hidden_state=None, input_ids=None, **kwargs):
        """Roll out one frame with the roles in parallel (ARCHITECTURE 5.4).

        Moshi walks all `2 * K` codebooks as one autoregressive sequence, so the user codebooks are
        conditioned on the assistant's within the frame. Haan instead stacks the roles on the
        **batch** axis and walks `K` steps, which is both ~2x shorter and the factorization
        ARCHITECTURE 5.4 argues for: `p(self_t | z_s) * p(user_t | z_s)`, each role reading the same
        context and neither seeing the other's frame-internal choice.

        The return layout is deliberately Moshi's -- `(batch, 1 + roles * K)`, the text token
        followed by the assistant's codebooks then the user's. `MoshiGenerationMixin` consumes it
        from two call sites and both keep working untouched.
        """
        roles = self.num_generation_roles
        codebooks = self.codebooks_per_role
        batch = input_ids.shape[0]

        # One row per (role, sequence) pair. `repeat_interleave` on the role index pairs with
        # `repeat` (tiling) on the rows below: rows `[0, batch)` are role 0, `[batch, 2 * batch)`
        # role 1 -- the order the reshape at the end unstacks.
        role_ids = torch.arange(roles, device=input_ids.device).repeat_interleave(batch)

        # How long the rollout is, is architecture, not a caller preference: exactly one step per
        # codebook of one stream plus the leading text token. Every knob that would change the step
        # count is dropped, not just the two `MoshiGenerationMixin.generate` happens to set -- it
        # sizes them for Moshi's `2 * K`-long sequence, and a survivor silently wins over the values
        # passed below and lands as an unreadable reshape error.
        for knob in ("min_length", "max_length", "min_new_tokens", "max_new_tokens", "max_time", "stop_strings"):
            kwargs.pop(knob, None)
        # These change the number of *rows* instead, which the role unstacking cannot absorb.
        for knob in ("num_return_sequences", "num_beams"):
            if kwargs.get(knob, 1) != 1:
                raise ValueError(
                    f"`{knob}` is not supported on the depth decoder: its rows are the role axis "
                    f"(ARCHITECTURE 5.4), so multiplying them would make role and sample indistinguishable. "
                    f"Pass it to the main `generate` instead."
                )

        outputs = super().generate(
            last_hidden_state=last_hidden_state.repeat(roles, 1, 1),
            input_ids=input_ids.repeat(roles, 1),
            role_ids=role_ids,
            min_length=codebooks + 1,
            max_length=codebooks + 1,
            **kwargs,
        )

        # Fail here rather than in the reshape, which reports only a shape and names nothing.
        if outputs.shape != (roles * batch, codebooks + 1):
            raise RuntimeError(
                f"the depth rollout produced {tuple(outputs.shape)}, expected "
                f"{(roles * batch, codebooks + 1)} ({roles} roles x {batch} rows, {codebooks} codebooks + "
                "the text token). Something overrode the rollout length or the batch size."
            )

        # (roles * batch, 1 + K) -> text token once, then each role's codebooks end to end.
        codes = outputs[:, 1:].reshape(roles, batch, codebooks)
        return torch.cat([outputs[:batch, :1], *codes], dim=1)

    def _split_roles(self, input_ids, labels, last_hidden_state):
        """Re-lay Moshi's sequential depth batch as the role-parallel one (ARCHITECTURE 5.4).

        `MoshiForConditionalGeneration.forward` builds one row per frame holding both streams end to
        end -- `[text, cb_0 .. cb_{roles*K-2}]`, `roles * K` wide. Walked as a single causal sequence
        that trains `p(user | z_s, self)`: the user positions attend to the assistant codebooks
        sitting in front of them. Generation samples `p(user | z_s)`, so left alone the user head
        would be trained on conditioning it never gets at inference.

        This turns that one row into `roles` rows of `K`, each `[text, cb_{r,0} .. cb_{r,K-2}]`, and
        tags them by role. Row order is role-major, matching `generate`. Returns `None` when the input
        is not that layout (already role-parallel, single-role, or a caller passing something else),
        in which case the sequential path runs unchanged.
        """
        roles, codebooks = self.config.num_roles, self.codebooks_per_role
        if roles < 2 or input_ids is None or input_ids.shape[1] != self.config.num_codebooks:
            return None
        if last_hidden_state is None or (labels is not None and labels.shape[1] != self.config.num_codebooks):
            return None

        text, audio_in = input_ids[:, :1], input_ids[:, 1:]
        frames = input_ids.shape[0]
        # Role `r` owns codebooks `[r*K, (r+1)*K)`. Its inputs are that stream's text token followed by
        # its own first `K-1` codebooks -- the last one is only ever a target, never an input.
        rows = [torch.cat([text, audio_in[:, r * codebooks : (r + 1) * codebooks - 1]], dim=1) for r in range(roles)]
        stacked_ids = torch.cat(rows, dim=0)

        stacked_labels = None
        if labels is not None:
            stacked_labels = labels.view(frames, roles, codebooks).transpose(0, 1).reshape(-1, codebooks)

        role_ids = torch.arange(roles, device=input_ids.device).repeat_interleave(frames)
        return stacked_ids, stacked_labels, last_hidden_state.repeat(roles, 1, 1), role_ids

    @can_return_tuple
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        last_hidden_state: torch.FloatTensor | None = None,
        attention_mask: torch.BoolTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        use_cache: bool | None = None,
        position_ids: torch.LongTensor | None = None,
        labels: torch.LongTensor | None = None,
        role_ids: torch.LongTensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> CausalLMOutputWithPast:
        r"""
        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens. The first element of the sequence must be the text token associated to
            the audio codebooks. The rest of the elements must be flattened audio codebooks.
        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            Sequence of hidden-states at the output of the last layer of the main decoder. Used to contextualize
            `input_ids`.
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the audio language modeling loss. Indices should either be in
            `[0, ..., config.audio_vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to
            `-100` are ignored (masked).
        """
        # Teacher-forced training arrives in Moshi's sequential layout -- one `roles * K`-long row per
        # frame, both streams end to end. Re-lay it as one `K`-long row per (frame, role) so training
        # uses the same factorization generation does (ARCHITECTURE 5.4); see `_split_roles`. Skipped
        # when the caller already supplies `role_ids` (the `generate` path, already role-parallel).
        stacked = None
        if role_ids is None and inputs_embeds is None and past_key_values is None:
            stacked = self._split_roles(input_ids, labels, last_hidden_state)
        if stacked is not None:
            # The split changes both axes -- `frames` rows of `2K` become `roles * frames` rows of
            # `K` -- so anything indexed by the OLD layout no longer describes this batch. Neither is
            # restacked: measured, a caller mask is dropped bit-for-bit (blanking half the sequence
            # left the logits identical to no mask at all), and a stale `position_ids` likewise. Both
            # reach here from the public API as `depth_decoder_attention_mask` / `_position_ids`.
            #
            # Rejected rather than restacked because no caller needs them: the depth sequence is
            # dense and causal over codebooks, Moshi's own path never passes either, and inventing a
            # remap would add surface that nothing exercises.
            for name, value in (("attention_mask", attention_mask), ("position_ids", position_ids)):
                if value is not None:
                    raise ValueError(
                        f"`{name}` cannot be combined with the role-parallel depth layout: the rows and the "
                        f"sequence length both change (ARCHITECTURE 5.4), so a {tuple(value.shape)} tensor "
                        f"built for the caller's layout would silently not apply. Drop it, or pass per-role "
                        "rows with `role_ids` and size it to those."
                    )
            input_ids, labels, last_hidden_state, role_ids = stacked
        elif (width := _sequence_width(input_ids, inputs_embeds)) > self.codebooks_per_role:
            # Anything wider than ONE stream that could not be re-laid. Rejected rather than run,
            # because the sequential walk it would fall back to is no longer a supported layout: it
            # trains `p(user | z_s, self)` where generation samples `p(user | z_s)`, and it indexes
            # an input table for slot `K - 1` that no longer exists (see `__init__`). Both failures
            # are quiet -- the first trains the wrong conditional, the second raises an `IndexError`
            # naming nothing.
            #
            # The bound is `> codebooks_per_role`, not `== num_codebooks`: guarding only the exact
            # `2K` width left every width between `K + 1` and `2K - 1` (a miscounted collator, say)
            # falling through to that same `IndexError`. Widths at or below `K` are legitimate --
            # a role-parallel row, or a single step during incremental decoding.
            reason = (
                "`role_ids` was supplied" if role_ids is not None
                else "`inputs_embeds` was supplied" if inputs_embeds is not None
                else "`past_key_values` was supplied" if past_key_values is not None
                else "`last_hidden_state` is missing, or `labels` does not match the input width"
            )
            raise ValueError(
                f"a {width}-wide depth sequence cannot be used here: {reason}, so it could not be re-laid as "
                f"one {self.codebooks_per_role}-wide row per role. Haan's depth decoder runs the roles in "
                "parallel over one stream each (ARCHITECTURE 5.4); pass either the full sequential batch on "
                "its own (`input_ids` + `last_hidden_state` + matching `labels`, no `role_ids`), or per-role "
                "rows together with `role_ids`."
            )

        # Same indices the body uses, computed before the backbone call advances `past_key_values`.
        past_seen_tokens = 0 if past_key_values is None else past_key_values.get_seq_length()
        slot_idx, _ = self.model._slot_and_role(get_codebook_idx(input_ids, inputs_embeds, past_seen_tokens))

        outputs: BaseModelOutputWithPast = self.model(
            input_ids=input_ids,
            last_hidden_state=last_hidden_state,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            position_ids=position_ids,
            role_ids=role_ids,
            **kwargs,
        )

        logits = self.lm_heads(outputs.last_hidden_state, slot_idx)

        loss = None
        if labels is not None:
            logits = logits.float()
            labels = labels.masked_fill(labels == self.config.audio_vocab_size, -100).reshape(-1)
            labels = labels.to(logits.device)
            loss = nn.functional.cross_entropy(logits.reshape(-1, self.config.audio_vocab_size), labels)

        if stacked is not None:
            # Hand the caller back the layout it passed in -- `(frames, roles * K, vocab)`, both streams
            # end to end -- so `MoshiForConditionalGeneration.forward` and everything reading
            # `audio_logits` is unaffected by the restructuring above.
            roles, frames = self.config.num_roles, logits.shape[0] // self.config.num_roles
            logits = logits.view(roles, frames, -1, logits.shape[-1]).transpose(0, 1).reshape(
                frames, -1, logits.shape[-1]
            )

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


class HaanForConditionalGeneration(HaanGenerationMixin, MoshiForConditionalGeneration):
    """The Haan model, for full-duplex speech-to-speech.

    Shared audio embeddings + role signal on the Temporal side (ARCHITECTURE 3.3), and a
    role-shared depth decoder (ARCHITECTURE 5.4). The delay pattern and the Mimi wiring are
    [`MoshiForConditionalGeneration`]'s, untouched; the generation-side deltas live in
    [`HaanGenerationMixin`], which is listed first so its hooks win over Moshi's.
    """

    config: HaanConfig
    # The BACKBONE's layers are `HaanDecoderLayer` (ARCHITECTURE 1's QK-Norm); the depth decoder's
    # are still `MoshiDecoderLayer`, and Moshi's own declaration covers those. transformers unions
    # the two -- `PreTrainedModel.__init__` walks the submodules and `update()`s their sets -- so
    # the effective value here is `{HaanDecoderLayer, MoshiDecoderLayer}`. This ADDS to the parent
    # list; replacing it would make the depth layers splittable again.
    #
    # Only accelerate reads this (`from_pretrained(device_map=...)`); the FSDP2 path in
    # `utils/train.py` matches by attribute path instead, which is why the gap went unnoticed.
    _no_split_modules = ["HaanDecoderLayer"]

    def __init__(self, config: HaanConfig):
        super().__init__(config)

        # ARCHITECTURE 3.3: `K` shared tables, where Moshi builds `2 * K` separate ones. Rebuilt on top of
        # the parent's construction rather than in place of it, so the rest of the parent `__init__` (the
        # depth decoder wiring, `predicts_user_stream`, the head) stays the single source of truth.
        self.embed_tokens = nn.ModuleList(
            [nn.Embedding(config.audio_vocab_size + 1, config.hidden_size) for _ in range(config.num_codebooks)]
        )
        self.role_embedding = RoleEmbedding(config.hidden_size, config.num_roles, config.role_mode)
        self.depth_decoder = HaanDepthDecoderForCausalLM._from_config(config.depth_decoder_config)

        # `MoshiForConditionalGeneration.__init__` hardcodes `self.model = MoshiModel(config)`, so the
        # backbone has to be replaced here too -- subclassing `MoshiModel` is not enough to get it used.
        # This was invisible while `HaanModel` was parameter-identical to its parent; it stops being
        # invisible now that `HaanDecoderLayer` carries QK-Norm (ARCHITECTURE 1), which would otherwise
        # be silently absent from every layer while `config.use_qk_norm` reported True.
        self.model = HaanModel(config)

        self.post_init()

    def forward(self, *, input_ids=None, assistant_audio_codes=None, user_audio_codes=None, inputs_embeds=None, **kwargs):
        """[`MoshiForConditionalGeneration.forward`], plus a text-only path for the anchor batches.

        RISKS 7.6 keeps a small pure-text loss running for the whole schedule -- it is the only guard
        against the Qwen3 backbone forgetting its multilingual ability, and `configs/data/loader.yaml`
        pins it at 5% with a hard floor because "if it silently drops out, Phase 4 is void". Those
        batches carry NO audio at all: `datasets/text_collator.py` refuses to fabricate silence to
        match shapes, since a 2048-token anchor would mean ~33k invented codes per sample.

        Moshi cannot take them. Its `forward` runs `torch.cat([assistant_audio_codes, user_audio_codes])`
        unconditionally, so two `None`s raise `TypeError` before reaching the `if audio_codes is not None`
        branch that would have handled them. Rather than copy the method to move one line, this embeds
        the text itself and hands the result over as `inputs_embeds`, which the parent already treats as
        taking priority over both other inputs -- the audio branch is then skipped rather than repaired.

        Going through the parent's `forward` (rather than reaching for `self.model` from the trainer) is
        what keeps this correct under FSDP2: `fully_shard` hangs its all-gather on the ROOT module's
        forward hook, so a text-only batch arriving first would otherwise read never-gathered
        `lm_head` / `embed_tokens` shards.
        """
        if inputs_embeds is None and assistant_audio_codes is None and user_audio_codes is None:
            if input_ids is None:
                raise ValueError("provide `input_ids`, `inputs_embeds`, or the audio codes.")
            inputs_embeds = self.model.embed_tokens(input_ids)
            input_ids = None  # the parent ignores it once `inputs_embeds` is set; do not imply otherwise

        return super().forward(
            input_ids=input_ids,
            assistant_audio_codes=assistant_audio_codes,
            user_audio_codes=user_audio_codes,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

In [ ]:
%%writefile project_amnesty/models/haan/generation_haan.py
"""Haan generation -- subclasses [`MoshiGenerationMixin`].

Mirrors the layout of `transformers/models/moshi/generation_moshi.py`: the audio
embedding hook and everything about how a frame is rolled out live here, not in
`modeling_haan.py`.

Two things differ from Moshi, and nothing else is touched -- the outer loop, the delay
pattern, the user-stream bookkeeping and `generate` itself are all inherited:

  - ARCHITECTURE 3.3  `_embed_audio_codes`: `K` shared tables + a role signal, where
    Moshi indexes `2 * K` separate ones. This is the only place audio codes become
    embeddings, for both training and generation.
  - ARCHITECTURE 5.4  the depth rollout runs the two roles as a **batch of 2 over
    `K` steps** instead of one sequential `2 * K`-step rollout, and 5.4's mode table
    (`set_depth_mode`) drops the user role entirely for live conversation.

The batch-2 rollout itself is implemented in `HaanDepthDecoderForCausalLM.generate`
(`modeling_haan.py`). That is deliberate: `MoshiGenerationMixin` calls
`self.depth_decoder.generate(...)` from two separate places -- once per step in
`prepare_inputs_for_generation`, once more for the trailing frame in `generate` -- and
both consume the result as `(batch, 1 + roles * K)`, text token first. Reassembling into
that exact layout inside the depth decoder keeps both call sites working untouched,
which is why neither of those two long methods is overridden here.
"""

import torch
from transformers.models.moshi.generation_moshi import MoshiGenerationMixin

__all__ = ["HaanGenerationMixin"]


class HaanGenerationMixin(MoshiGenerationMixin):
    """Generation loop for [`HaanForConditionalGeneration`]."""

    def _embed_audio_codes(self, audio_codes: torch.Tensor) -> torch.Tensor:
        """Sum the per-codebook audio embeddings, marking each stream with its role.

        `audio_codes` spans both streams, the assistant's codebooks followed by the user's (the
        layout `_split_predicted_streams` documents). Moshi reads that straight off `2 * K` tables;
        Haan folds it onto `K` shared tables -- codebook `c` uses table `c % K` -- and recovers the
        stream from `c // K` (ARCHITECTURE 3.3).

        Like Moshi's, each embedding's output is moved to a single device before summing, so a
        `ModuleList` split across a device map still works.
        """
        target_device = self.embed_tokens[0].weight.device
        codebooks_per_role = self.num_codebooks

        embeds = None
        for codebook in range(audio_codes.shape[1]):
            slot, role = codebook % codebooks_per_role, codebook // codebooks_per_role
            codebook_embeds = self.embed_tokens[slot](audio_codes[:, codebook]).to(target_device)
            codebook_embeds = self.role_embedding(codebook_embeds, role)
            embeds = codebook_embeds if embeds is None else embeds + codebook_embeds

        return embeds

    def set_depth_mode(self, mode: str) -> None:
        """Switch the depth decoder between the two modes of ARCHITECTURE 5.4.

        | mode           | predicts          | depth batch | use                                                |
        |----------------|-------------------|-------------|----------------------------------------------------|
        | `"live"`       | self only (q8)    | 1           | real conversation -- the user stream is real input |
        | `"simulation"` | self + user (q16) | 2           | offline eval, synthetic dialogue                   |

        ARCHITECTURE 5.0.3: the user prediction is a training target, but in a live conversation the
        actual user audio is used instead and the prediction is thrown away. Rolling it out anyway
        costs a second batch element for an output nobody reads, so `"live"` skips it outright.

        Training always runs both roles -- it goes through `forward`, not `generate`, and does not
        consult this switch.
        """
        if mode not in ("live", "simulation"):
            raise ValueError(f"`mode={mode!r}` must be 'live' or 'simulation'.")

        roles = 1 if mode == "live" else self.config.num_roles
        self.depth_decoder.num_generation_roles = roles
        # `_split_predicted_streams` slices the user half off the depth decoder's output, so it has to
        # agree with what was actually rolled out. Left alone, "live" would hand it a self-only tensor
        # and it would slice an empty user stream out of thin air.
        self.predicts_user_stream = roles > 1

In [ ]:
%%writefile project_amnesty/models/haan/processing_haan.py
"""Haan processor -- subclasses the Moshi processor from `transformers`.

Inherits everything from [`MoshiProcessor`] (Mimi codec, dual audio streams, text
padded to one token per audio frame) and changes exactly one thing: which id fills the
text stream between enunciations.

Scope note. The Zone A / B / C instruction template (ARCHITECTURE 7.2) is *not* built
here -- `MOSHI_KD_DATASET_PLAN.md` 10.9 assigns that to the collator, which assembles
`[Zone A | Zone B | Zone C]` per batch so the voice prompt can vary per epoch. What the
processor owns is the token contract those layers share.
"""

from transformers.models.moshi.processing_moshi import MoshiProcessor

__all__ = ["HaanProcessor"]


class HaanProcessor(MoshiProcessor):
    r"""Mimi codec + Qwen3 tokenizer, with Moshi's stream PAD kept distinct from batch padding.

    ARCHITECTURE 7.6 and RISKS 7.12 both turn on one distinction that Moshi's tokenizers
    make for free and Qwen3's does not:

      - **stream PAD** -- "not speaking, session continues". It fills roughly 65% of the
        text channel, is a prediction target, and is down-weighted (x0.3) rather than
        masked.
      - **batch pad** -- length alignment across a batch. Masked out of the loss entirely.

    Qwen3 has no spare `<pad>`, and HF convention points `pad_token_id` at `<|im_end|>`.
    Inheriting `MoshiProcessor._pad_token_id` unchanged would therefore fill the stream
    with the *end-of-message* token -- which is ARCHITECTURE 7.6's exact failure: a token
    that means "generation over" would come to occupy 65% of the text channel and
    overwrite the instruction-following behaviour the Qwen3 backbone was chosen for.

    So the stream PAD is passed in explicitly. It must be a newly assigned reserved slot,
    never `<|im_end|>` / `<|im_start|>`.

    Args:
        stream_pad_id (`int`, *optional*):
            Id of the stream PAD token (`configs/tokens.yaml: text_pad_id`).
        stream_epad_id (`int`, *optional*):
            Id of the EPAD token (`configs/tokens.yaml: text_epad_id`), inserted one frame
            before a word starts. Carried here so the collator and the processor read one
            source of truth; the processor itself does not insert it, since EPAD placement
            needs word-level timestamps.
    """

    def __init__(
        self,
        feature_extractor,
        tokenizer,
        audio_tokenizer,
        num_codebooks=8,
        stream_pad_id: int | None = None,
        stream_epad_id: int | None = None,
    ):
        self.stream_pad_id = stream_pad_id
        self.stream_epad_id = stream_epad_id
        super().__init__(feature_extractor, tokenizer, audio_tokenizer, num_codebooks=num_codebooks)

    def _pad_token_id(self) -> int:
        """The id used to fill the text stream between enunciations.

        Overrides Moshi's `pad_token_id` -> `<pad>` lookup, which on a Qwen3 tokenizer would
        resolve to the end-of-message token. Fails loudly instead of falling back: a silently
        wrong id here does not crash, it just quietly trains the wrong turn-boundary behaviour
        (RISKS 7.12, 7.13).
        """
        if self.stream_pad_id is None:
            raise ValueError(
                "`stream_pad_id` is unset, so the text stream cannot be padded. Moshi's stream PAD is a "
                "distinct token from the batch pad (ARCHITECTURE 7.6): assign it to a reserved Qwen3 slot and "
                "pass it here, rather than reusing `<|im_end|>`/`<|im_start|>`. Source of truth: "
                "`configs/tokens.yaml: text_pad_id`."
            )
        banned = self._banned_stream_pad_ids()
        if self.stream_pad_id in banned:
            raise ValueError(
                f"`stream_pad_id={self.stream_pad_id}` is {banned[self.stream_pad_id]}, which cannot also be the "
                "stream PAD. The stream PAD marks 'not speaking, session continues' and fills ~65% of the text "
                "channel as a down-weighted prediction target; these tokens mean 'this is over' and are either "
                "masked out of the loss (batch pad) or terminate generation. Reusing one overwrites that "
                "association across most of the channel and damages the very instruction-following the Qwen3 "
                "backbone was chosen for (ARCHITECTURE 7.6, RISKS 7.12/7.13). Assign a reserved slot instead."
            )

        if self.stream_pad_id < 0:
            raise ValueError(f"`stream_pad_id={self.stream_pad_id}` must be non-negative.")

        # No UPPER bound is checked here, deliberately. The stream PAD is meant to live in a
        # reserved slot -- an embedding row that exists but carries no token (Qwen3-8B has 151936
        # rows for 151669 tokens, and `configs/tokens.yaml` uses three of that gap). Only the MODEL
        # knows how many rows there are, and a processor must not have to be told: it is a data-side
        # object, and reaching for `config.vocab_size` here would invert the dependency. Bounding by
        # `len(tokenizer)` instead is worse than not checking -- it rejects every reserved slot that
        # exists, i.e. exactly what the docstring above tells the caller to assign.
        #
        # What the tokenizer alone CAN settle is the direction that actually corrupts training:
        # below `len(tokenizer)` the id is a token the backbone was already trained on, and filling
        # ~65% of the text channel with it overwrites whatever it meant. Same damage as reusing
        # `<|im_end|>`, only quieter, since a rare token gives no clue at review time.
        real_tokens = len(self.tokenizer)
        if self.stream_pad_id < real_tokens:
            token = self.tokenizer.convert_ids_to_tokens(self.stream_pad_id)
            raise ValueError(
                f"`stream_pad_id={self.stream_pad_id}` is the existing token {token!r}, not a reserved slot. "
                "The stream PAD occupies most of the text channel, so that token's learned meaning would be "
                f"overwritten (ARCHITECTURE 7.6). Use an id at or above {real_tokens} -- an embedding row that "
                "exists but carries no token. Source of truth: `configs/tokens.yaml`."
            )
        return self.stream_pad_id

    def _banned_stream_pad_ids(self) -> dict[int, str]:
        """Ids that must never serve as the stream PAD, mapped to why.

        Resolved from the tokenizer rather than hardcoded: `<|im_end|>` is 151645 on Qwen3 and
        something else everywhere else, so a literal would silently stop guarding the moment the
        tokenizer changed. `convert_tokens_to_ids` returns None for a token the vocabulary does not
        have, so a missing marker simply drops out of the set.
        """
        banned: dict[int, str] = {}
        for attribute, reason in (
            ("pad_token_id", "the tokenizer's batch `pad_token_id`"),
            ("eos_token_id", "the tokenizer's `eos_token_id`"),
        ):
            token_id = getattr(self.tokenizer, attribute, None)
            if token_id is not None:
                banned.setdefault(token_id, reason)

        # ARCHITECTURE 7.6 names these two explicitly. Kept separate from the eos lookup above
        # because they coincide on Qwen3 (`eos == <|im_end|>`) and need not elsewhere -- and
        # because naming the marker makes the error say what the caller actually did.
        for marker in ("<|im_start|>", "<|im_end|>"):
            token_id = self.tokenizer.convert_tokens_to_ids(marker)
            if token_id is not None:
                banned.setdefault(token_id, f"the ChatML marker `{marker}`")
        return banned

In [ ]:
%%writefile project_amnesty/models/haan/__init__.py
from .configuration_haan import HaanConfig, HaanDepthConfig
from .generation_haan import HaanGenerationMixin
from .modeling_haan import (
    HaanDepthDecoderForCausalLM,
    HaanDepthDecoderModel,
    HaanForConditionalGeneration,
    HaanModel,
    RoleEmbedding,
)
from .processing_haan import HaanProcessor

__all__ = [
    "HaanConfig",
    "HaanDepthConfig",
    "HaanModel",
    "HaanDepthDecoderModel",
    "HaanDepthDecoderForCausalLM",
    "RoleEmbedding",
    "HaanGenerationMixin",
    "HaanForConditionalGeneration",
    "HaanProcessor",
]

In [ ]:
%%writefile project_amnesty/utils/warm_start_haan.py
"""Dual-source warm-start for Haan (ARCHITECTURE 1 / 5.4.1 / 5.4.2).

Haan is assembled from two pretrained models, because its two halves come from different places:

    model.* / lm_head / self_attn.{q,k}_norm     <- Qwen3      (ARCHITECTURE 1)
    embed_tokens.{0..K-1}                        <- Moshi, user half   (ARCHITECTURE 5.4.2)
    depth_decoder.*                              <- Moshi      (ARCHITECTURE 5.4.1)

**Why the Qwen3 side needs surgery and the Moshi side does not.** Haan's backbone is
`HaanModel(MoshiModel)`, so it is laid out like Moshi even while holding Qwen3's weights. The two
describe the same transformer with different module shapes:

  - Qwen3 keeps `gate_proj` and `up_proj` separate; `MoshiGatingMLP` fuses them into one `fc1` and
    splits the result in half at run time. So `fc1 = cat([gate, up], dim=0)`, **gate first** --
    `MoshiGatingMLP.forward` does `view(..., 2, -1)` and activates `h[..., 0, :]`, which is the
    contiguous first half. Swapping the halves is silent and trains a model whose gate and value
    are exchanged. `ffn_dim` must be `2 * intermediate_size` for the fused shape to exist at all.
  - `MoshiLinear` wraps `nn.Linear` as `.linear`, so `q_proj.weight` -> `q_proj.linear.weight`.
  - `MoshiModel` allocates `vocab_size + 1` embedding rows -- one reserved id used as the text
    stream's audio-only/BOS position. Qwen3 has no such row, so it is appended, not loaded.
  - Qwen3's QK-Norm has a home only because `HaanAttention` adds `q_norm`/`k_norm`
    (`config.use_qk_norm`); a stock `MoshiAttention` would drop 2 tensors per layer.

Everything else is a rename or a straight copy. The mapping was verified end to end: a tiny
`Qwen3Model` and a `HaanModel` built through it produce bit-identical fp32 output.

**What starts cold.** `depth_decoder.model.text_embed_tokens` (~155M parameters) has no source at
all -- Moshi's is sized for its own 32k tokenizer and the rows do not correspond to Qwen3's. That,
plus the reserved embedding row and the two role embeddings, is ~1.7% of the model (the depth text
table dominates the figure). Everything unsourced is printed rather than allowed to pass as
transferred.

Location note: this module lives under `utils/` rather than `models/haan/` so that `models/` never
imports from `utils/`. It is model *assembly*, not model *definition* -- the classes in
`models/haan/` stay independent of where their initial weights come from.
"""

from __future__ import annotations

import re

import torch

__all__ = [
    "INIT_SOURCES",
    "UNSOURCED_BY_DESIGN",
    "haan_config_from_moshi",
    "haan_config_from_qwen3_and_moshi",
    "moshi_audio_tables",
    "warm_start_from_moshi",
    "warm_start_qwen3_moshi",
]

# ARCHITECTURE 5.4.2. Which half of Moshi's `2 * K` audio tables seeds Haan's `K` shared ones.
#
#   "user"   emb[K:2K] -- the DEFAULT and the documented choice. Moshi's user stream was trained
#            with the second speaker's voice resampled per example, so this half never collapsed
#            onto a single actor. Removing fixed-speaker bias is this project's premise, which
#            makes it the better prior for voice-prompt cloning (5.2).
#   "self"   emb[0:K]  -- Moshi's own stream, narrowed to one actor's voice by instruct tuning.
#            Kept for the 3.6 ablation.
#   "random" leave Haan's own init; measures what the warm-start is actually worth.
#
# The choice is NOT cosmetic and cannot be left to name matching: Moshi's tables 0..7 carry the
# same NAMES as Haan's 0..7, so a naive load_state_dict silently takes the "self" half.
INIT_SOURCES = ("user", "self", "random")

# Moshi audio tables are `embed_tokens.<i>.weight` in HF format. Anchored at the start so it
# matches neither `model.embed_tokens.weight` (the backbone's text table) nor
# `depth_decoder.model.embed_tokens.N.weight`.
_AUDIO_TABLE_RE = re.compile(r"^embed_tokens\.(\d+)\.weight$")

# Backbone projections whose Moshi name carries the `MoshiLinear` wrapper. Restricted to
# `model.layers.` on purpose: the depth decoder has leaves with the identical final name whose
# weights are rank-3 `MoshiFlexibleLinear` tensors and must never receive a 2-D Qwen3 tensor.
_PROJ_RENAME_RE = re.compile(r"^(model\.layers\.\d+\.self_attn\.[qkvo]_proj)\.weight$")
_GATE_UP_RE = re.compile(r"^(model\.layers\.\d+)\.mlp\.(gate|up)_proj\.weight$")
_DOWN_RE = re.compile(r"^(model\.layers\.\d+)\.mlp\.down_proj\.weight$")

# Haan parameters that no source was ever going to supply, under the Qwen3+Moshi assembly.
# Enumerated rather than inferred so that a genuine shape mismatch still fails loudly instead of
# being absorbed as "expected" -- in particular `text_embed_tokens`, whose Moshi counterpart has
# the right NAME and the wrong SHAPE and would otherwise abort the whole warm-start.
UNSOURCED_BY_DESIGN = (
    # Moshi's is (32001, depth_hidden), sized for its own tokenizer; the rows do not correspond to
    # Qwen3's vocabulary. Not sliceable, not remappable -- this is the one large cold block.
    "depth_decoder.model.text_embed_tokens.weight",
)


# ======================================================================================
# Config construction
# ======================================================================================

def haan_config_from_moshi(
    moshi_config,
    *,
    num_roles: int = 2,
    role_mode: str = "scale",
    predict_user_stream: bool = True,
    use_qk_norm: bool = False,
    **overrides,
):
    """Derive a `HaanConfig` describing a MOSHI-backbone Haan.

    Used by the Moshi-only warm-start (the ARCHITECTURE 3.6 baseline arm). Everything dimensional
    is Moshi's, unchanged -- 5.4.1 reuses audio cardinality 2048 and depth dim 1024 as-is.

    `use_qk_norm` defaults to **False** here, and that default is load bearing: `HaanConfig`'s own
    default is True (Haan's normal shape is a Qwen3 backbone) and `MoshiConfig.to_dict()` carries
    no `use_qk_norm` key, so without passing it explicitly every Moshi warm-start would silently
    build QK-Norm the Moshi weights never saw. An all-ones RMSNorm still rescales its input, so
    that is not a harmless extra module -- it breaks the transfer at every layer's q and k.
    """
    from project_amnesty.models.haan.configuration_haan import HaanConfig  # noqa: PLC0415

    config_dict = moshi_config.to_dict()
    config_dict.pop("model_type", None)
    config_dict.pop("architectures", None)

    depth_dict = dict(config_dict.get("depth_decoder_config") or {})
    depth_dict.pop("model_type", None)
    codebooks_per_role = int(config_dict["num_codebooks"])
    depth_dict["num_codebooks"] = num_roles * codebooks_per_role if predict_user_stream else codebooks_per_role

    config_dict["depth_decoder_config"] = depth_dict
    config_dict["num_roles"] = num_roles
    config_dict["role_mode"] = role_mode
    config_dict["use_qk_norm"] = use_qk_norm
    # `moshi_config.to_dict()` carries Moshi's own `sliding_window=3000`, which would override
    # `HaanConfig`'s deliberate None. Cleared so this arm differs from the Qwen3 arm ONLY in the
    # backbone -- it exists to isolate the Qwen3 substitution (ARCH 3.6), and a context limit
    # present in one arm and absent in the other is a confound. Harmless in practice: a 3000-frame
    # window is bit-identical to full causal below 3000, and this arm is capped at Moshi's
    # `max_position_embeddings` anyway.
    config_dict["sliding_window"] = None
    config_dict.update(overrides)

    return HaanConfig(**config_dict)


def haan_config_from_qwen3_and_moshi(
    qwen3_config,
    moshi_config,
    *,
    num_roles: int = 2,
    role_mode: str = "scale",
    predict_user_stream: bool = True,
    **overrides,
):
    """Derive a `HaanConfig` describing a QWEN3-backbone Haan with Moshi's audio stack.

    Backbone dimensions come from Qwen3; the audio, Mimi and depth sub-configs from Moshi. Four
    fields are handled specially because getting them wrong is silent rather than fatal:

      - **`rope_parameters`** is set as a whole dict, never as a bare `rope_theta=`. Layering
        `rope_theta=1_000_000` onto a Moshi-derived config does nothing: the dict already carries
        `rope_parameters={"rope_theta": 10000.0}` and transformers resolves it with `setdefault`,
        so the existing value wins and the override is dropped with no warning at all.
      - **`pad_token_id`** is forced to None. `MoshiModel` passes it straight into
        `nn.Embedding(..., padding_idx=)`, so a real id there zeroes that row AND pins its gradient
        to zero permanently. Moshi keeps its BOS/pad ids in the GenerationConfig instead.
      - **`ffn_dim`** is `2 * intermediate_size`, because `MoshiGatingMLP` fuses gate and up into
        one `fc1`. Leaving Moshi's own value makes both MLP projections fail to load.
      - **`sliding_window`** is cleared to None. Qwen3 is full-attention, and Moshi's inherited
        3000 is not inert: `MoshiModel.forward` selects `create_sliding_window_causal_mask`
        whenever it is non-None, and the cache builds window layers that physically evict KV. It
        is bit-identical to full causal up to 3000 positions, so it passes every short test and
        diverges only on long context -- which is exactly why it has to be set here rather than
        noticed later. It must also be set BEFORE any cache is constructed.
    """
    from project_amnesty.models.haan.configuration_haan import HaanConfig  # noqa: PLC0415

    qwen3 = qwen3_config.to_dict()
    moshi = moshi_config.to_dict()

    head_dim = qwen3.get("head_dim") or qwen3["hidden_size"] // qwen3["num_attention_heads"]
    rope_theta = (qwen3.get("rope_parameters") or {}).get("rope_theta") or qwen3.get("rope_theta")

    config_dict = {
        # --- backbone: Qwen3 (ARCHITECTURE 1) ---
        "vocab_size": qwen3["vocab_size"],
        "hidden_size": qwen3["hidden_size"],
        "num_hidden_layers": qwen3["num_hidden_layers"],
        "num_attention_heads": qwen3["num_attention_heads"],
        "num_key_value_heads": qwen3["num_key_value_heads"],
        # Explicit: MoshiConfig derives head_dim as hidden_size // num_attention_heads when unset,
        # which happens to be right for Qwen3-8B (4096/32) and is not right in general.
        "head_dim": head_dim,
        "hidden_act": qwen3.get("hidden_act", "silu"),
        "max_position_embeddings": qwen3["max_position_embeddings"],
        "rms_norm_eps": qwen3["rms_norm_eps"],
        "ffn_dim": 2 * qwen3["intermediate_size"],
        "rope_parameters": {"rope_type": "default", "rope_theta": rope_theta},
        "initializer_range": qwen3.get("initializer_range", 0.02),
        "attention_dropout": qwen3.get("attention_dropout", 0.0),
        # Impossible to tie here regardless: embed_tokens has one row lm_head does not.
        "tie_word_embeddings": False,
        "use_cache": qwen3.get("use_cache", True),
        "pad_token_id": None,
        "sliding_window": None,
        "use_qk_norm": True,
        # --- audio stack: Moshi (ARCHITECTURE 4 / 5.4) ---
        "audio_vocab_size": moshi["audio_vocab_size"],
        "num_codebooks": moshi["num_codebooks"],
        "audio_encoder_config": moshi.get("audio_encoder_config"),
        "num_roles": num_roles,
        "role_mode": role_mode,
    }

    # The depth decoder is Moshi's end to end, so it keeps Moshi's dimensions AND Moshi's
    # `rms_norm_eps` -- only the backbone becomes Qwen3.
    depth_dict = dict(moshi.get("depth_decoder_config") or {})
    depth_dict.pop("model_type", None)
    codebooks_per_role = int(moshi["num_codebooks"])
    depth_dict["num_codebooks"] = num_roles * codebooks_per_role if predict_user_stream else codebooks_per_role
    config_dict["depth_decoder_config"] = depth_dict

    config_dict.update(overrides)
    return HaanConfig(**config_dict)


# ======================================================================================
# Checkpoint reading
# ======================================================================================

def moshi_audio_tables(
    moshi_ckpt: str,
    *,
    num_codebooks: int = 8,
    side: str = "user",
    revision: str | None = None,
) -> list["torch.Tensor"]:
    """Read Moshi's audio input embedding tables straight out of the checkpoint shards.

    Returns `num_codebooks` tensors of shape `(audio_vocab_size + 1, hidden_size)` -- the half of
    Moshi's `2 * K` tables named by `side` ("user" -> `[K, 2K)`, "self" -> `[0, K)`).

    Reads only the `embed_tokens.*` tensors via safetensors' lazy `safe_open`, so the backbone is
    never materialized. This is the same quantity the warm-start copies in, which is why
    `utils/evaluate.py` measures the ARCHITECTURE 5.4.2 / RISKS 3 embedding drift against THIS
    function rather than re-deriving the initialization independently.

    HF-format checkpoints only (`kmhf/hf-moshiko` and friends), where the tables are
    `embed_tokens.<i>.weight`. The original Kyutai release names them `emb.<i>.weight`;
    `project_amnesty/tools/inspect_moshi_weights.py` reads that layout instead.
    """
    if side not in ("user", "self"):
        raise ValueError(f"`side={side!r}` must be 'user' or 'self'.")

    from safetensors import safe_open  # noqa: PLC0415

    wanted_start = num_codebooks if side == "user" else 0
    wanted = {wanted_start + k: k for k in range(num_codebooks)}  # checkpoint index -> Haan slot

    tables: dict[int, torch.Tensor] = {}
    for shard in _resolve_shards(moshi_ckpt, _AUDIO_TABLE_RE, revision=revision):
        with safe_open(shard, framework="pt") as f:
            for key in f.keys():
                match = _AUDIO_TABLE_RE.match(key)
                if match is not None and int(match.group(1)) in wanted:
                    tables[wanted[int(match.group(1))]] = f.get_tensor(key)

    missing = sorted(set(range(num_codebooks)) - set(tables))
    if missing:
        raise KeyError(
            f"{moshi_ckpt} is missing the {side}-side audio tables for codebooks {missing} "
            f"(looked for embed_tokens.{{{wanted_start}..{wanted_start + num_codebooks - 1}}}.weight). "
            "Is this an HF-format Moshi checkpoint?"
        )
    return [tables[k] for k in range(num_codebooks)]


def _resolve_shards(ckpt: str, key_filter: "re.Pattern | None" = None, *, revision: str | None = None) -> list[str]:
    """Local paths of a checkpoint's safetensors shards.

    `key_filter` restricts the result to shards that actually hold a matching tensor, which for a
    cold cache also skips downloading the rest; pass None to take every shard.
    """
    import json  # noqa: PLC0415
    import os  # noqa: PLC0415

    def _wanted(weight_map: dict) -> list[str]:
        return sorted({s for k, s in weight_map.items() if key_filter is None or key_filter.match(k)})

    if os.path.isdir(ckpt):
        index_path = os.path.join(ckpt, "model.safetensors.index.json")
        single = os.path.join(ckpt, "model.safetensors")
        if os.path.isfile(index_path):
            with open(index_path, encoding="utf-8") as f:
                return [os.path.join(ckpt, name) for name in _wanted(json.load(f)["weight_map"])]
        if os.path.isfile(single):
            return [single]
        raise FileNotFoundError(f"no safetensors weights under {ckpt}")

    from huggingface_hub import hf_hub_download  # noqa: PLC0415
    from huggingface_hub.errors import EntryNotFoundError  # noqa: PLC0415

    try:
        index_path = hf_hub_download(repo_id=ckpt, filename="model.safetensors.index.json", revision=revision)
    except (EntryNotFoundError, OSError):
        # Unsharded checkpoint: a single model.safetensors and no index.
        return [hf_hub_download(repo_id=ckpt, filename="model.safetensors", revision=revision)]

    with open(index_path, encoding="utf-8") as f:
        weight_map: dict[str, str] = json.load(f)["weight_map"]
    return [hf_hub_download(repo_id=ckpt, filename=name, revision=revision) for name in _wanted(weight_map)]


def _stream_tensors(ckpt: str, *, revision: str | None = None):
    """Yield `(name, tensor)` for every tensor in a checkpoint, one shard at a time.

    Streamed rather than loaded through `Qwen3ForCausalLM.from_pretrained` because holding Qwen3,
    Moshi and Haan resident at once is ~50 GB in bf16. Safe to read raw for Qwen3: unlike Moshi --
    whose released checkpoints need transformers' `_checkpoint_conversion_mapping` to reach the
    current module layout -- Qwen3's on-disk keys already match its modules.
    """
    from safetensors import safe_open  # noqa: PLC0415

    for shard in _resolve_shards(ckpt, revision=revision):
        with safe_open(shard, framework="pt") as f:
            for key in f.keys():
                yield key, f.get_tensor(key)


# ======================================================================================
# Warm-start entry points
# ======================================================================================

def warm_start_qwen3_moshi(
    qwen3_ckpt: str = "Qwen/Qwen3-8B",
    moshi_ckpt: str = "kmhf/hf-moshiko",
    *,
    init_source: str = "user",
    num_roles: int = 2,
    role_mode: str = "scale",
    predict_user_stream: bool = True,
    dtype: "torch.dtype | None" = None,
    revision: str | None = None,
    verbose: bool = True,
    **config_overrides,
):
    """Assemble a Haan model: Qwen3 backbone + Moshi audio embeddings + Moshi depth decoder.

    ARCHITECTURE 1 + 5.4.1 + 5.4.2 as one operation. See the module docstring for the full mapping
    and for what starts cold.

    Args:
        qwen3_ckpt: Qwen3 checkpoint for the Temporal backbone, `lm_head` and QK-Norm.
        moshi_ckpt: Moshi checkpoint for the audio tables and the whole depth decoder.
        init_source: which half of Moshi's audio tables seeds the shared ones (`INIT_SOURCES`).
        num_roles / role_mode / predict_user_stream: forwarded to the config builder.
        dtype: dtype to build Haan as. Defaults to Qwen3's checkpoint dtype.
        verbose: print the transfer summary, including everything that did NOT transfer.
        **config_overrides: forwarded onto the `HaanConfig`.

    Returns:
        `HaanForConditionalGeneration`, warm-started.
    """
    from transformers import AutoConfig  # noqa: PLC0415
    from transformers.models.moshi.configuration_moshi import MoshiConfig  # noqa: PLC0415

    from project_amnesty.models.haan.modeling_haan import HaanForConditionalGeneration  # noqa: PLC0415

    if init_source not in INIT_SOURCES:
        raise ValueError(f"`init_source={init_source!r}` must be one of {INIT_SOURCES}.")

    qwen3_config = AutoConfig.from_pretrained(qwen3_ckpt, revision=revision)
    moshi_config = MoshiConfig.from_pretrained(moshi_ckpt)
    config = haan_config_from_qwen3_and_moshi(
        qwen3_config,
        moshi_config,
        num_roles=num_roles,
        role_mode=role_mode,
        predict_user_stream=predict_user_stream,
        **config_overrides,
    )

    dtype = dtype or getattr(qwen3_config, "dtype", None) or getattr(qwen3_config, "torch_dtype", None)
    model = HaanForConditionalGeneration._from_config(config, dtype=dtype)

    summary = _new_summary()
    _copy_qwen3_backbone(qwen3_ckpt, model, summary, revision=revision)
    _copy_moshi_audio_and_depth(moshi_ckpt, model, summary, init_source=init_source, dtype=dtype)

    _finalize(model, summary)
    if verbose:
        _report(summary, sources=f"{qwen3_ckpt} (backbone) + {moshi_ckpt} (audio/depth)", init_source=init_source)
    return model


def warm_start_from_moshi(
    moshi_ckpt: str = "kmhf/hf-moshiko",
    *,
    init_source: str = "user",
    num_roles: int = 2,
    role_mode: str = "scale",
    predict_user_stream: bool = True,
    dtype: "torch.dtype | None" = None,
    revision: str | None = None,
    verbose: bool = True,
    **config_overrides,
):
    """Assemble a Haan model warm-started ENTIRELY from Moshi -- backbone included.

    The ARCHITECTURE 3.6 baseline arm: the same audio/Depth deltas, but Helium rather than Qwen3
    behind them, so the Korean-emergence comparison has a control. Built with `use_qk_norm=False`,
    since a Moshi backbone has no QK-Norm to load.

    Memory: both models are resident while the tensors are copied (~2x the checkpoint). The Moshi
    side is released before returning. Load in bf16 (`dtype=torch.bfloat16`) to halve it.
    """
    from transformers.models.moshi.modeling_moshi import MoshiForConditionalGeneration  # noqa: PLC0415

    from project_amnesty.models.haan.modeling_haan import HaanForConditionalGeneration  # noqa: PLC0415

    if init_source not in INIT_SOURCES:
        raise ValueError(f"`init_source={init_source!r}` must be one of {INIT_SOURCES}.")

    # Loaded through the class (rather than raw shards) on purpose: transformers'
    # `_checkpoint_conversion_mapping` for Moshi rewrites the released checkpoints' older key
    # layout (`decoder.model.*`, `depth_decoder.layers.*`) onto the current module paths. Reading
    # raw tensors would reimplement that mapping, and drift from it on the next upstream change.
    moshi = MoshiForConditionalGeneration.from_pretrained(moshi_ckpt, dtype=dtype, revision=revision)
    try:
        config = haan_config_from_moshi(
            moshi.config,
            num_roles=num_roles,
            role_mode=role_mode,
            predict_user_stream=predict_user_stream,
            use_qk_norm=False,
            **config_overrides,
        )
        model = HaanForConditionalGeneration._from_config(config, dtype=dtype or moshi.dtype)
        source = dict(moshi.named_parameters())
        source.update(moshi.named_buffers())
        summary = _new_summary()
        _copy_named(source, model, summary, init_source=init_source)
    finally:
        del moshi

    _finalize(model, summary)
    if verbose:
        _report(summary, sources=moshi_ckpt, init_source=init_source)
    return model


# ======================================================================================
# Copy machinery
# ======================================================================================

def _new_summary() -> dict:
    return {"copied": [], "kept": [], "mismatched": [], "unused": [], "notes": []}


@torch.no_grad()
def _copy_qwen3_backbone(qwen3_ckpt: str, haan, summary: dict, *, revision: str | None = None) -> None:
    """Stream Qwen3's parameters into Haan's Moshi-shaped backbone."""
    targets = _targets(haan)

    # gate/up fuse into one `fc1`, so they are buffered until both halves are in hand.
    mlp_halves: dict[str, dict[str, torch.Tensor]] = {}
    embed_source: torch.Tensor | None = None
    seen: set[str] = set()

    for name, tensor in _stream_tensors(qwen3_ckpt, revision=revision):
        seen.add(name)

        if name == "model.embed_tokens.weight":
            embed_source = tensor
            continue

        gate_up = _GATE_UP_RE.match(name)
        if gate_up is not None:
            mlp_halves.setdefault(gate_up.group(1), {})[gate_up.group(2)] = tensor
            continue

        target_name = _DOWN_RE.sub(r"\1.mlp.fc2.weight", name)
        target_name = _PROJ_RENAME_RE.sub(r"\1.linear.weight", target_name)
        _assign(targets, target_name, tensor, summary, source_name=name)

    # --- fused MLP: gate FIRST (see the module docstring; the order is not recoverable later) ---
    for layer, halves in sorted(mlp_halves.items()):
        if set(halves) != {"gate", "up"}:
            summary["mismatched"].append(f"{layer}.mlp: expected gate_proj and up_proj, got {sorted(halves)}")
            continue
        _assign(
            targets,
            f"{layer}.mlp.fc1.weight",
            torch.cat([halves["gate"], halves["up"]], dim=0),
            summary,
            source_name=f"{layer}.mlp.{{gate,up}}_proj.weight",
        )

    # --- text embedding: Qwen3's rows, plus Moshi's one reserved row ---
    # `MoshiModel` allocates `vocab_size + 1` rows; the extra id is the text stream's audio-only /
    # BOS position and has no Qwen3 counterpart. Seeded from the mean of the real rows so it starts
    # in-distribution rather than at whatever the random init produced.
    target = targets.get("model.embed_tokens.weight")
    if embed_source is None:
        summary["kept"].append("model.embed_tokens.weight")
    elif target is None:
        summary["unused"].append("model.embed_tokens.weight")
    elif tuple(target.shape) != (embed_source.shape[0] + 1, embed_source.shape[1]):
        summary["mismatched"].append(
            f"model.embed_tokens.weight: haan {tuple(target.shape)} != qwen3 {tuple(embed_source.shape)} + 1 row"
        )
    else:
        source = embed_source.to(device=target.device, dtype=target.dtype)
        target[: source.shape[0]].copy_(source)
        target[source.shape[0]].copy_(source.mean(dim=0))
        summary["copied"].append("model.embed_tokens.weight")
        summary["notes"].append(
            f"model.embed_tokens.weight: rows [0:{source.shape[0]}) from Qwen3; reserved row "
            f"{source.shape[0]} (Moshi's audio-only/BOS id) seeded with the row mean -- no Qwen3 source."
        )


@torch.no_grad()
def _copy_moshi_audio_and_depth(
    moshi_ckpt: str, haan, summary: dict, *, init_source: str, dtype: "torch.dtype | None" = None
) -> None:
    """Copy Moshi's audio tables and its whole depth decoder into Haan, leaving the backbone alone."""
    from transformers.models.moshi.modeling_moshi import MoshiForConditionalGeneration  # noqa: PLC0415

    moshi = MoshiForConditionalGeneration.from_pretrained(moshi_ckpt, dtype=dtype)
    try:
        source = {
            name: tensor
            for name, tensor in moshi.named_parameters()
            if _AUDIO_TABLE_RE.match(name) or name.startswith("depth_decoder.")
        }
        _copy_named(source, haan, summary, init_source=init_source, audio_and_depth_only=True)
    finally:
        del moshi


@torch.no_grad()
def _copy_named(
    source: dict, haan, summary: dict, *, init_source: str, audio_and_depth_only: bool = False
) -> None:
    """Name-matched copy with the one audio remap applied.

    `audio_and_depth_only=True` restricts the walk to the parameters a Moshi source is responsible
    for in the dual-source assembly, leaving the Qwen3-filled backbone untouched.
    """
    codebooks_per_role = int(haan.config.num_codebooks)
    audio_offset = codebooks_per_role if init_source == "user" else 0

    targets = _targets(haan)
    consumed: set[str] = set()

    for name, target in targets.items():
        audio_table = _AUDIO_TABLE_RE.match(name)
        if audio_and_depth_only and audio_table is None and not name.startswith("depth_decoder."):
            continue

        if audio_table is not None:
            # The one remap: Haan's shared table `k` takes Moshi's `k + offset`. Same NAME space as
            # Moshi's assistant half, so this must not fall through to the identity branch.
            if init_source == "random":
                summary["kept"].append(name)
                continue
            source_name = f"embed_tokens.{int(audio_table.group(1)) + audio_offset}.weight"
        else:
            source_name = name

        tensor = source.get(source_name)
        if tensor is None:
            summary["kept"].append(name)
            continue
        consumed.add(source_name)
        _assign(targets, name, tensor, summary, source_name=source_name)

    summary["unused"].extend(sorted(set(source) - consumed))


def _targets(haan) -> dict:
    targets = dict(haan.named_parameters())
    targets.update(haan.named_buffers())
    return targets


def _assign(targets: dict, name: str, tensor: torch.Tensor, summary: dict, *, source_name: str) -> None:
    """Copy one tensor into `targets[name]`, recording the outcome."""
    target = targets.get(name)
    if target is None:
        summary["unused"].append(source_name)
        return
    if target.shape != tensor.shape:
        # Not raised here: the walk finishes first so `_finalize` can report every mismatch at
        # once, and so an expected-no-source name (UNSOURCED_BY_DESIGN) can be reclassified.
        summary["mismatched"].append(
            f"{name}: haan {tuple(target.shape)} != source {source_name} {tuple(tensor.shape)}"
        )
        return
    target.copy_(tensor.to(device=target.device, dtype=target.dtype))
    summary["copied"].append(name)


def _finalize(model, summary: dict) -> None:
    """Reclassify the by-design gaps, complete the `kept` set, then fail on real mismatches."""
    present = _targets(model)
    copied = set(summary["copied"])

    for name in UNSOURCED_BY_DESIGN:
        if name not in present or name in copied:
            continue
        # A same-name/different-shape source landed in `mismatched`; that is the expected outcome
        # for these, not a broken checkpoint. Drop those entries and record the gap instead.
        summary["mismatched"] = [m for m in summary["mismatched"] if not m.startswith(f"{name}:")]

    # `kept` is derived here rather than accumulated during the walk, because no single walk sees
    # every parameter: the Qwen3 pass iterates its own SOURCE tensors, and the Moshi pass is scoped
    # to the audio tables and the depth decoder. A parameter belonging to neither -- notably the
    # Temporal `role_embedding` -- was therefore correctly left at its init and silently omitted
    # from the report. Since the report exists precisely to name what did not transfer, it is
    # computed as "everything not copied" instead.
    mismatched_names = {m.split(":", 1)[0] for m in summary["mismatched"]}
    summary["kept"] = sorted(
        name
        for name in present
        if name not in copied and name not in mismatched_names and not _is_derived(name)
    )

    if summary["mismatched"]:
        # A surviving mismatch means the checkpoint does not describe this architecture -- the
        # result would be neither warm-started nor cleanly random, so it is an error.
        raise ValueError(
            "warm-start shape mismatch (a source checkpoint does not match this Haan config):\n  "
            + "\n  ".join(sorted(set(summary["mismatched"])))
        )


def _is_derived(name: str) -> bool:
    """Buffers reconstructed from the config at build time, so never 'missing' from a checkpoint.

    `rotary_emb.inv_freq` / `original_inv_freq` are a pure function of `rope_parameters` and
    `head_dim`, and Qwen3 does not ship them (they are non-persistent). Listing them alongside the
    genuinely cold parameters would bury the ones that matter.
    """
    return ".rotary_emb." in f".{name}"


def _report(summary: dict, *, sources: str, init_source: str) -> None:
    """Print what did and did not transfer. The 'kept' list is the point of this function."""
    kept = sorted(set(summary["kept"]))
    print(
        f"[haan] warm-start from {sources}: {len(set(summary['copied']))} tensors copied "
        f"(audio init_source={init_source!r}).",
        flush=True,
    )
    for note in summary["notes"]:
        print(f"[haan]   {note}", flush=True)
    if kept:
        print(f"[haan] {len(kept)} parameter(s) had NO source and keep their own init:", flush=True)
        for name in kept:
            print(f"[haan]   {name}", flush=True)
    if summary["unused"]:
        print(
            f"[haan] {len(set(summary['unused']))} source tensor(s) unused "
            f"(expected: the opposite-side audio tables).",
            flush=True,
        )

## 3. 임포트

In [ ]:
import sys
for m in [k for k in list(sys.modules) if k.startswith("project_amnesty")]:
    del sys.modules[m]
from project_amnesty.utils.warm_start_haan import warm_start_qwen3_moshi, INIT_SOURCES
from project_amnesty.models.haan import HaanForConditionalGeneration
import torch
print("import OK | init_source 선택지:", INIT_SOURCES)

## 4. 빠른 검증 — 조립 로직이 bit-exact 인가 (합성 소스, 수 초)

실제 9B 를 받기 전에, warm-start 가 **백본을 Qwen3 로 정확히 재현**하고 **오디오는 Moshi user 측을** 가져오는지
작은 합성 체크포인트로 먼저 확인한다. 32GB 를 받지 않고도 조립 로직 자체를 검증할 수 있다.

In [ ]:
import tempfile
from transformers import Qwen3Config, Qwen3ForCausalLM, MoshiConfig, MoshiForConditionalGeneration

K = 8
tmp = pathlib.Path(tempfile.mkdtemp())
torch.manual_seed(0)
Qwen3ForCausalLM(Qwen3Config(vocab_size=140, hidden_size=64, num_hidden_layers=2,
    num_attention_heads=4, num_key_value_heads=2, head_dim=16, intermediate_size=64,
    max_position_embeddings=128, rms_norm_eps=1e-6)).save_pretrained(tmp/"qwen3")
MoshiForConditionalGeneration(MoshiConfig(hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
    ffn_dim=128, vocab_size=60, num_codebooks=K, audio_vocab_size=2048,
    depth_decoder_config={'hidden_size':32,'num_hidden_layers':2,'num_attention_heads':4,
                          'ffn_dim':64,'num_codebooks':K})).save_pretrained(tmp/"moshi")

haan = warm_start_qwen3_moshi(str(tmp/"qwen3"), str(tmp/"moshi"), init_source="user", verbose=True)

from transformers import Qwen3ForCausalLM as Q, MoshiForConditionalGeneration as Mo
q = Q.from_pretrained(tmp/"qwen3").eval(); mo = Mo.from_pretrained(tmp/"moshi")
ids = torch.randint(0, 140, (2, 6))
with torch.no_grad():
    d = (q.model(input_ids=ids).last_hidden_state - haan.eval().model(input_ids=ids).last_hidden_state).abs().max().item()
print(f"\n백본이 Qwen3 를 재현: max|Δ| = {d:.2e}   {'bit-exact' if d==0 else '불일치'}")
au = torch.equal(haan.embed_tokens[0].weight, mo.state_dict()['embed_tokens.8.weight'])
print(f"공유 오디오 emb[0] == Moshi emb[K] (user 측): {au}")

## 5. 실제 warm-start — Qwen3-8B 백본 + Moshiko 오디오/Depth

여기가 본론이다. **백본만 Qwen3-8B 로 교체하고, 오디오/Depth 는 `kmhf/hf-moshiko` 가중치를 그대로** 얹는다.

- 첫 실행 시 Qwen3-8B(~16GB)를 받는다. Moshi 는 캐시되어 있으면 재사용.
- `verbose=True` 가 무엇이 복사되고 무엇이 cold(소스 없음)로 남는지 전부 출력한다.
- 조립된 모델은 bf16 로 ~18GB. GPU 없이 CPU 로도 조립된다(느릴 뿐).

> 실행 시간과 용량이 크다. 스킵하려면 이 셀을 건너뛰어도 위 4번이 조립 로직을 이미 증명한다.

In [ ]:
REAL = True   # 실제 9B 조립을 원치 않으면 False

if REAL:
    model = warm_start_qwen3_moshi(
        "Qwen/Qwen3-8B", "kmhf/hf-moshiko",
        init_source="user", dtype=torch.bfloat16, verbose=True,
    )
    n = sum(p.numel() for p in model.parameters())
    print(f"\n조립 완료: {n/1e9:.2f}B params, dtype {next(model.parameters()).dtype}")
else:
    model = None
    print("REAL=False -> 실제 조립 건너뜀")

## 6. 조립된 모델이 실제로 도는가 (forward + generate)

In [ ]:
if model is not None:
    model.eval()
    cfg = model.config; K = cfg.num_codebooks
    B, T = 1, 4
    ids = torch.randint(0, cfg.vocab_size, (B, T))
    s   = torch.randint(0, cfg.audio_vocab_size, (B, K, T))
    u   = torch.randint(0, cfg.audio_vocab_size, (B, K, T))
    dev = next(model.parameters()).device
    with torch.no_grad():
        out = model(input_ids=ids.to(dev), assistant_audio_codes=s.to(dev), user_audio_codes=u.to(dev))
    print("forward OK | text_logits", tuple(out.logits.shape))
    g = model.generate(input_ids=ids.to(dev), assistant_audio_codes=s.to(dev), user_audio_codes=u.to(dev),
                       max_new_tokens=4, do_sample=False, return_audio_codes=True)
    print("generate OK | audio", tuple(g.audio_codes.shape))
else:
    print("model 이 없어(5번 건너뜀) 이 셀은 스킵")

## 7. 요약

- **4번**: 조립 로직이 bit-exact — 백본이 Qwen3 를 정확히 재현, 오디오는 Moshi user 측. (합성, 항상 실행 가능)
- **5번**: 실제 Qwen3-8B 백본 + Moshiko 오디오/Depth 로 9B Haan 조립. `verbose` 로그가 cold 파라미터(text_embed_tokens ~155M, role emb 2개, 예약행 ≈ 1.7%)를 명시.
- **6번**: 조립된 모델이 forward/generate 정상.

warm-start 가 **무엇을 어디서** 가져오는지는 `warm_start_haan.py` 모듈 docstring 참조.